# Sensitivitätsanalyse – Portfolio-Layer Grid Search

**Dieses Notebook ist ein reiner Launcher.** Es klont das Git-Repository und führt
`run_sensitivity.py` direkt aus – kein manuelles Aktualisieren von Zellen nötig.

**Benötigte Datasets (Add data → Your datasets):**
- `busersteven/trading-results` — enthält Checkpoints + Metadaten
- `busersteven/trading-raw-data` — enthält die Parquet-Kursdaten

**Accelerator:** CPU reicht (kein Training, nur Inferenz + Portfolio-Simulation)

**Laufzeit:** ~30–60 Min (Score-Cache) + ~5 Min (Grid Search + IC-Chart)

| Parameter | Werte |
|---|---|
| `n_max` | 5, 7, 9 |
| `rotation_buffer` | 2, 3, 4 |
| `hard_stop_pct` | 20 %, 25 %, 30 % |
| `fees` | 0.1 %, 0.15 %, 0.2 % |

In [ ]:
# ── Konfiguration ─────────────────────────────────────────────────────────────
# Nur diese Zelle muss bei Bedarf angepasst werden.

REPO_URL    = 'https://github.com/stevenlangeshops/trading.git'
REPO_BRANCH = 'main'
HORIZON     = 7          # Vorhersage-Horizont in Handelstagen
DEVICE      = 'cpu'      # 'cpu' oder 'cuda'

# Score-Cache: Pfad zum Speichern / Laden des vorberechneten Score-Caches.
# - Erster Lauf: Cache wird berechnet (~30-60 Min) und unter diesem Pfad gespeichert.
# - Weitere Läufe: Cache wird in Sekunden geladen statt neu berechnet.
# - Auf None setzen, um den Cache nie zu persistieren.
SCORE_CACHE_PATH = '/kaggle/working/score_cache.parquet'

# Policy-Vergleich (Baseline/A1/A2/A3/B) laeuft immer automatisch.
# Kein Flag mehr noetig – wird direkt in der Analyse-Zelle aktiviert.

# Sensitivity-Ergebnisse werden in ein EIGENES Dataset hochgeladen,
# damit trading-results (mit Checkpoints) NIEMALS überschrieben wird.
KAGGLE_USERNAME         = 'busersteven'
SENSITIVITY_DATASET_ID  = f'{KAGGLE_USERNAME}/trading-sensitivity'

In [ ]:
# ── Setup: Repo klonen, Dependencies, Artefakte kopieren ─────────────────────
# Diese Zelle ist statisch und muss nie geändert werden.

import json, os, shutil, subprocess, sys, tarfile
from pathlib import Path

WORKING  = Path('/kaggle/working')
REPO_DIR = WORKING / 'repo'
CKPT_DIR = WORKING / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

# ── Repo klonen ───────────────────────────────────────────────────────────────
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ['git', 'clone', '--depth=1', '-b', REPO_BRANCH, REPO_URL, str(REPO_DIR)],
    check=True,
)
print(f'Repo geklont: {REPO_BRANCH}')

# ── Dependencies ─────────────────────────────────────────────────────────────
subprocess.run(
    [sys.executable, '-m', 'pip', 'install',
     'ta==0.11.0', 'loguru==0.7.2', '--quiet', '--no-warn-script-location'],
    check=True,
)
print('Dependencies ok')

# ── Checkpoints + JSON-Artefakte kopieren ────────────────────────────────────
# Debug: zeige was im Input-Verzeichnis liegt
print('Inhalt /kaggle/input:')
for p in sorted(Path('/kaggle/input').rglob('*'))[:40]:
    print(f'  {p}')

# Suche nach beliebigen fold_*_best.pt (nicht nur fold_0)
pt_files = sorted(Path('/kaggle/input').rglob('fold_*_best.pt'))
if pt_files:
    src_dir = pt_files[0].parent
    for f in src_dir.iterdir():
        shutil.copy(f, CKPT_DIR / f.name)
    print(f'\nArtefakte aus {src_dir}/ kopiert ({len(pt_files)} .pt Dateien)')
else:
    tar = next((p for p in Path('/kaggle/input').rglob('kaggle_artifacts.tar.gz')), None)
    assert tar, (
        'Weder fold_*_best.pt noch kaggle_artifacts.tar.gz gefunden!\n'
        'trading-results Dataset hinzufügen (Add data → Your datasets → busersteven/trading-results)'
    )
    with tarfile.open(tar) as tf:
        tf.extractall(str(CKPT_DIR))
    # ggf. eine Verschachtelungsebene entfernen
    nested_pt = sorted(CKPT_DIR.rglob('fold_*_best.pt'))
    if nested_pt and nested_pt[0].parent != CKPT_DIR:
        for f in nested_pt[0].parent.iterdir():
            shutil.move(str(f), str(CKPT_DIR / f.name))
    print(f'Artefakte aus tar.gz entpackt')

print('Checkpoints:', sorted(p.name for p in CKPT_DIR.glob('*.pt')))

# ── Score-Cache aus Dataset laden (falls vorhanden) ───────────────────────────
cached_sc = next((p for p in Path('/kaggle/input').rglob('score_cache.parquet')), None)
if cached_sc and SCORE_CACHE_PATH:
    shutil.copy(cached_sc, SCORE_CACHE_PATH)
    size_mb = Path(SCORE_CACHE_PATH).stat().st_size / 1024 / 1024
    print(f'Score-Cache aus Dataset geladen: {SCORE_CACHE_PATH}  ({size_mb:.1f} MB) → wird nicht neu berechnet')
else:
    print('Kein Score-Cache im Dataset gefunden → wird beim ersten Lauf neu berechnet und gespeichert')

# ── Parquet-Kursdaten ins Repo-Unterverzeichnis kopieren ─────────────────────
raw_dest = REPO_DIR / 'data' / 'raw'
raw_dest.mkdir(parents=True, exist_ok=True)
parquets = list(Path('/kaggle/input').rglob('*.parquet'))
assert parquets, 'Keine Parquet-Dateien – trading-raw-data Dataset hinzufügen!'
for f in parquets:
    shutil.copy(f, raw_dest / f.name)
print(f'{len(parquets)} Parquet-Dateien nach data/raw/ kopiert')

# ── Pfade auflösen ────────────────────────────────────────────────────────────
wf_json   = next(CKPT_DIR.rglob(f'v2_{HORIZON}d_walk_forward.json'))
asset_map = next(CKPT_DIR.rglob('asset_map.json'))
print(f'walk_forward JSON : {wf_json.name}')
print(f'asset_map         : {asset_map.name}')

In [ ]:
# ── Analyse ausführen ─────────────────────────────────────────────────────────
# run_sensitivity.py wird direkt als Subprocess aufgerufen.
# Alle Änderungen im Git sind sofort aktiv – diese Zelle bleibt unverändert.

cmd = [
    sys.executable, str(REPO_DIR / 'run_sensitivity.py'),
    '--ckpt-dir',   str(CKPT_DIR),
    '--walk-json',  str(wf_json),
    '--asset-map',  str(asset_map),
    '--data-dir',   str(raw_dest),
    '--output',     str(WORKING / 'sensitivity_results.csv'),
    '--plot',       str(WORKING / 'sensitivity_top_equity.png'),
    '--ic-plot',    str(WORKING / 'rolling_ic.png'),
    '--horizon',    str(HORIZON),
    '--device',     DEVICE,
    '--repo-dir',   str(REPO_DIR),
    '--policy-csv',     str(WORKING / 'policy_comparison.csv'),
    '--policy-plot',    str(WORKING / 'policy_equity.png'),
    '--policy-compare',  # immer aktiv – Baseline + A1/A2/A3/B (IC20/IC30/IC40/SPY200)
]
if SCORE_CACHE_PATH:
    cmd += ['--score-cache', SCORE_CACHE_PATH]

print('Starte run_sensitivity.py ...\n' + ' '.join(cmd) + '\n')

# stdout/stderr strömen direkt in die Zelle (kein capture)
result = subprocess.run(cmd, cwd=str(REPO_DIR))

if result.returncode != 0:
    print(f'\n[WARNUNG] run_sensitivity.py exit code {result.returncode}')
    print('Grid-Search-Ergebnisse (CSV/Equity-Chart) sind trotzdem nutzbar.')
    print('Scrolle nach oben für den vollständigen Traceback.')
else:
    print('\n✓ Analyse abgeschlossen')

In [ ]:
# ── Ergebnisse anzeigen ───────────────────────────────────────────────────────

import pandas as pd
from IPython.display import Image, display

import pandas as pd
from IPython.display import Image, display

# ── Grid Search Ergebnisse ────────────────────────────────────────────────────
csv_path = WORKING / 'sensitivity_results.csv'
if csv_path.exists():
    results = pd.read_csv(str(csv_path), index_col=0)
    print('=== Top-20 Konfigurationen nach Sharpe ===')
    display(
        results.head(20).style
        .background_gradient(subset=['sharpe'],          cmap='Greens')
        .background_gradient(subset=['total_return_%'],  cmap='Blues')
        .background_gradient(subset=['max_drawdown_%'],  cmap='Reds_r')
        .format({
            'hard_stop_pct':  '{:.0%}',
            'fees':           '{:.3%}',
            'sharpe':         '{:.3f}',
            'total_return_%': '{:+.1f}%',
            'max_drawdown_%': '{:.1f}%',
            'win_rate_%':     '{:.1f}%',
        })
    )

# ── Charts ────────────────────────────────────────────────────────────────────
for title, fname in [
    ('Top-5 Equity-Kurven (Sensitivitätsanalyse)', 'sensitivity_top_equity.png'),
    ('Rolling Rank-IC – täglicher & monatlicher IC', 'rolling_ic.png'),
    ('Policy-Vergleich Equity-Kurven', 'policy_equity.png'),
]:
    p = WORKING / fname
    if p.exists():
        print(f'\n=== {title} ===')
        display(Image(str(p)))
    else:
        print(f'\n[FEHLT] {fname}')

# ── Policy-Vergleich Tabelle ──────────────────────────────────────────────────
pc_path = WORKING / 'policy_comparison.csv'
if pc_path.exists():
    pc = pd.read_csv(str(pc_path))
    print('\n=== Policy-Vergleich: Baseline vs. A1/A2/A3/B ===')
    display(
        pc.style
        .background_gradient(subset=['sharpe'],         cmap='Greens')
        .background_gradient(subset=['total_return_%'], cmap='Blues')
        .background_gradient(subset=['dd_2022_%'],      cmap='Reds_r')
        .background_gradient(subset=['dd_2025_%'],      cmap='Reds_r')
        .format({
            'sharpe':             '{:.3f}',
            'total_return_%':     '{:+.1f}%',
            'max_drawdown_%':     '{:.1f}%',
            'win_rate_%':         '{:.1f}%',
            'pct_days_reduced':   '{:.1f}%',
            'ret_2022_%':         '{:+.1f}%',
            'dd_2022_%':          '{:.1f}%',
            'ret_2023_%':         '{:+.1f}%',
            'ret_2024_%':         '{:+.1f}%',
            'ret_2025_%':         '{:+.1f}%',
            'dd_2025_%':          '{:.1f}%',
        })
    )

In [ ]:
# ── Ergebnisse in EIGENEM Dataset speichern (trading-sensitivity) ─────────────
# WICHTIG: Ergebnisse gehen NICHT nach trading-results, damit Checkpoints
#          dort niemals überschrieben werden.
import time

kaggle_key = None
try:
    from kaggle_secrets import UserSecretsClient
    kaggle_key = UserSecretsClient().get_secret('KAGGLE_KEY')
except Exception:
    pass

upload_files = [
    'sensitivity_results.csv', 'sensitivity_top_equity.png',
    'rolling_ic.png', f'rolling_ic_v2_{HORIZON}d.json', f'rolling_ic_v2_{HORIZON}d.csv',
    'policy_comparison.csv', 'policy_equity.png',
]
if SCORE_CACHE_PATH:
    upload_files.append(Path(SCORE_CACHE_PATH).name)

if kaggle_key:
    cfg = Path('/root/.kaggle/kaggle.json')
    cfg.parent.mkdir(parents=True, exist_ok=True)
    cfg.write_text(json.dumps({'username': KAGGLE_USERNAME, 'key': kaggle_key}))
    cfg.chmod(0o600)

    up = WORKING / 'sensitivity_upload'
    up.mkdir(exist_ok=True)
    for fname in upload_files:
        src = Path(SCORE_CACHE_PATH) if fname.endswith('.parquet') else WORKING / fname
        if src.exists():
            shutil.copy(src, up / fname)

    meta = up / 'dataset-metadata.json'
    # Prüfen ob Dataset bereits existiert → version; sonst → create
    check = subprocess.run(
        ['kaggle', 'datasets', 'status', SENSITIVITY_DATASET_ID],
        capture_output=True, text=True,
    )
    meta.write_text(json.dumps({
        'title': 'trading-sensitivity',
        'id': SENSITIVITY_DATASET_ID,
        'licenses': [{'name': 'other'}],
    }))
    if check.returncode == 0:
        r = subprocess.run(
            ['kaggle', 'datasets', 'version', '-p', str(up),
             '-m', f'Sensitivity {time.strftime("%Y%m%d_%H%M%S")}', '--dir-mode', 'zip'],
            capture_output=True, text=True,
        )
    else:
        r = subprocess.run(
            ['kaggle', 'datasets', 'create', '-p', str(up), '--dir-mode', 'zip'],
            capture_output=True, text=True,
        )
    print(r.stdout or r.stderr)
else:
    print('Kein KAGGLE_KEY – Dateien manuell herunterladen:')
    for fname in upload_files:
        src = Path(SCORE_CACHE_PATH) if fname.endswith('.parquet') else WORKING / fname
        if src.exists():
            size_mb = src.stat().st_size / 1024 / 1024
            print(f'  /kaggle/working/{fname}  ({size_mb:.1f} MB)')